In [24]:
## Import necessary libraries
import json
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown
from datasets import load_dataset
from loguru import logger
import nnsight
from reasoners.lm import HFModel
from qwen_math_parser import math_equal

from transformers import StoppingCriteria, StoppingCriteriaList

In [2]:
# Configure prettier plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12
sns.set_palette("bright")

In [3]:
## Configuration settings
# Model paths (adjust these to your environment)
POLICY_MODEL_PATH = "/data/manikya/Llama-3.2-1B-Instruct"
REWARD_MODEL_PATH = "/data/manikya/math-shepherd-mistral-7b-prm"
CACHE_DIR = "/data/manikya/huggingface"
PROMPT_PATH = "/home/manikya/llm-reasoners/examples/Inference-Scaling-SGL/math500/prompts.json"  # Path to your prompts JSON file

In [5]:
## Setup logging
logger.remove()
logger.add(lambda msg: print(msg, end=""), colorize=True, level="INFO")

1

# Helper functions for loading models and data

In [4]:
policy_model = nnsight.LanguageModel(POLICY_MODEL_PATH, device_map="cuda:0")
reward_model = HFModel(model_pth=REWARD_MODEL_PATH, tokenizer_pth=REWARD_MODEL_PATH, device="cuda:0")


You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

In [6]:
def load_math_problem(problem_id=None, problem_idx=0, cache_dir=CACHE_DIR):
    """Load a specific math problem by ID or index."""
    dataset = load_dataset("HuggingFaceH4/MATH-500", split="test", cache_dir=cache_dir)
    
    if problem_id:
        for example in dataset:
            if example.get("id") == problem_id:
                return example
        raise ValueError(f"Problem with ID {problem_id} not found")
    else:
        return dataset[problem_idx]

In [7]:
def load_prompt_template(prompt_path=PROMPT_PATH):
    """Load the prompt template for math problems."""
    with open(prompt_path, "r") as f:
        return json.load(f)

In [8]:
def format_prompt(problem_text, steps=None, prompt_template=None):
    """Format the problem with optional solution steps."""
    if prompt_template is None:
        prompt_template = load_prompt_template()
    
    # Replace placeholders in the template
    prompt = prompt_template["icl"].replace("<init_state>", problem_text).replace("<problem_state>", "")
    
    # Add existing solution steps if provided
    if steps and len(steps) > 0:
        steps_text = "\n".join([f"Step {i+1}: {step}" for i, step in enumerate(steps)])
        prompt += "\n" + steps_text + "\n"
        prompt += f"\nStep {len(steps)+1}:"
    
    return prompt

In [9]:
def eval_answer(ground_truth, predicted):
    """Evaluate if the predicted answer matches the ground truth."""
    if not predicted or not ground_truth:
        return False
    ground_truth = ground_truth.replace(" ", "").lower()
    predicted = predicted.replace(" ", "").lower()
    return math_equal(ground_truth, predicted)

# Core steering functions

In [14]:
def calculate_steering_vector(good_activations, bad_activations, strategy="reward_weighted", scale=1.0):
    """Calculate steering vectors using different strategies."""
    steering_vectors = {}
    
    # Process each layer
    for layer_idx in good_activations:
        steering_vectors[layer_idx] = {}
        
        # Process each component within the layer
        for component in good_activations[layer_idx]:
            if component not in bad_activations[layer_idx]:
                continue
            
            good_acts = good_activations[layer_idx][component]
            bad_acts = bad_activations[layer_idx][component]
            
            # Skip if tensor shapes don't match
            if good_acts.shape != bad_acts.shape:
                logger.warning(f"Shape mismatch in layer {layer_idx}, component {component}: {good_acts.shape} vs {bad_acts.shape}")
                continue
            
            # Calculate the steering vector based on the selected strategy
            if strategy == "reward_weighted":
                # Weight by reward difference
                steering_vector = scale * (good_acts - bad_acts)
            elif strategy == "best_only":
                # Use only the best example
                steering_vector = scale * good_acts
            elif strategy == "contrast":
                # Emphasize differences: move toward good and away from bad
                steering_vector = scale * (2 * good_acts - bad_acts)
            else:
                raise ValueError(f"Unknown steering strategy: {strategy}")
            
            steering_vectors[layer_idx][component] = steering_vector
    
    return steering_vectors

In [16]:
def get_layer_component_combinations():
    """Get predefined combinations of layers and components to test."""
    combos = [
        {
            "name": "Last 3 layers, output projection",
            "layers": [13, 14, 15],
            "components": ["self_attn.o_proj"]
        },
        {
            "name": "Last 3 layers, MLP",
            "layers": [13, 14, 15],
            "components": ["mlp.up_proj", "mlp.down_proj"]
        },
        {
            "name": "Last 3 layers, QKV",
            "layers": [13, 14, 15],
            "components": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj"]
        },
        {
            "name": "Middle layers, output projection",
            "layers": [6, 7, 8],
            "components": ["self_attn.o_proj"]
        },
        {
            "name": "All attention components in last layer",
            "layers": [15],
            "components": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj"]
        }
    ]
    return combos

# Exp

In [56]:
import re

def extract_steps(solution_text, problem_statement=None):
    """
    Extract the steps from a solution text, ignoring example steps in the prompt.
    
    Args:
        solution_text (str): The full text containing the solution
        problem_statement (str, optional): The problem statement to help locate where
                                          the actual solution begins
    
    Returns:
        list: A list of the extracted steps
    """
    # Find all step patterns in the text
    step_pattern = re.compile(r'## Step \d+:.*?(?=## Step \d+:|Therefore, the final answer|$)', 
                             re.DOTALL)
    
    # If we have a problem statement, use it to find where the actual solution starts
    if problem_statement:
        # Find where the problem statement occurs in the text
        problem_pos = solution_text.find(problem_statement)
        if problem_pos != -1:
            # Only look at the text after the problem statement
            solution_text = solution_text[problem_pos + len(problem_statement):]
    
    # Extract steps from the filtered text
    steps = step_pattern.findall(solution_text)
    
    # Find the conclusion separately
    conclusion_pattern = re.compile(r'Therefore, the final answer.*?(?=## Step \d+:|$)', re.DOTALL)
    conclusion = conclusion_pattern.search(solution_text)
    
    # If there's a conclusion and at least one step, add it to the last step
    if conclusion and steps:
        steps[-1] = steps[-1] + conclusion.group(0)
    elif conclusion:  # If there's only a conclusion but no steps
        steps.append(conclusion.group(0))
    
    # Clean up the steps (strip extra whitespace)
    steps = [step.strip() for step in steps]
    
    return steps

In [63]:
def reward(
        steps,
        original_prompt,
    ) -> float:
    # Define tokens for the reward model
    good_token = "+"
    bad_token = "-"
    step_tag = "ки"

    # Prepare the current problem state
    current_problem_state = "\n".join(
        [f"Step {step.strip()} {step_tag}" for step in steps]
    )

    # Create the input for the reward model
    input_for_prm = f"{original_prompt} \n {current_problem_state}"

    input_id = torch.tensor([reward_model.tokenizer.encode(input_for_prm)])

    candidate_tokens = reward_model.tokenizer.encode(f"{good_token} {bad_token}")[1:]
    step_tag_id = reward_model.tokenizer.encode(f"{step_tag}")[-1]

    with torch.no_grad():
        logits = reward_model.model(input_id).logits[:, :, candidate_tokens]
        scores = logits.softmax(dim=-1)[:, :, 0] 
        step_scores = scores[input_id == step_tag_id]
    
    intuition = step_scores[-1].item()
    
    return intuition, step_scores

In [15]:
prompt_template = load_prompt_template()

In [146]:
def compute_steering_vector_contrast(
        positive_activations,
        negative_activations,
        layers_to_use,
        components_to_use,
        steering_config 
    ):
    """
    Compute steering vectors using a contrastive approach.
    
    Args:
        positive_activations: List of activations for positive examples
        negative_activations: List of activations for negative examples
        device: Device to use for computation
        
    Returns:
        Dictionary of steering vectors for each layer
    """
    steering_vectors = {}
    if any(layer < 0 for layer in layers_to_use):
        # Dynamically determine the layers to use based on model architecture
        # Get total number of layers from first activation dict
        first_layer_name = list(positive_activations[0].keys())[0]  # Get any layer name
        total_layers = int(first_layer_name.split(".")[2])  # Extract layer number
        
        # Convert negative indices to positive
        layers_to_use = [layer if layer >= 0 else total_layers + layer + 1 for layer in layers_to_use]
        logger.info(f"Dynamically determined layers to use: {layers_to_use}")

    
    steering_scale = steering_config["steering_scale"]
    
    # Process layer weights
    layer_weights = steering_config.get("layer_specific_scales", {}) if steering_config["use_layer_specific_scaling"] else {}
    if not layer_weights:
        layer_weights = {layer: 1.0 for layer in layers_to_use}
        
    # Process component weights
    component_weights = steering_config.get("component_weights", {})
    if not component_weights:
        component_weights = {component: 1.0 for component in components_to_use}
    
    # Calculate steering vectors for each layer and component
    for layer_name in positive_activations[0].keys():
        # Parse layer path to get layer index and component
        parts = layer_name.split(".")
        layer_idx = int(parts[2])
        component = ".".join(parts[3:])
        
        # Skip if this layer or component is not in our configured list
        if layer_idx not in layers_to_use or component not in components_to_use:
            continue
            
        # Get layer and component weights
        layer_weight = layer_weights.get(str(layer_idx), 1.0)  # Convert to string for YAML compatibility
        component_weight = component_weights.get(component, 1.0)
        
        # Stack activations for this layer from positive and negative examples
        pos_activations = torch.stack([act[layer_name] for act in positive_activations])
        neg_activations = torch.stack([act[layer_name] for act in negative_activations])
        
        # Compute mean activations
        pos_mean = torch.mean(pos_activations, dim=0)
        neg_mean = torch.mean(neg_activations, dim=0)
        
        # Compute steering vector as the difference between positive and negative means
        steering_vector = pos_mean - neg_mean
        
        # Normalize if enabled
        if steering_config["normalize_vectors"]:
            steering_vector = steering_vector / torch.norm(steering_vector, dim=-1, keepdim=True)
        
        # Apply layer, component, and global scaling factors
        steering_vector = steering_vector * layer_weight * component_weight * steering_scale
        
        steering_vectors[layer_name] = steering_vector
            
    return steering_vectors

In [ ]:
class StopStringCriteria(StoppingCriteria):
            def __init__(self, tokenizer, stop_string, prompt):
                self.tokenizer = tokenizer
                self.stop_string = stop_string
                self.prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids[0]
                self.prompt_len = len(self.prompt_ids)
                
            def __call__(self, input_ids, scores, **kwargs):
                # Get the generated text (excluding the prompt)
                if input_ids.shape[1] <= self.prompt_len:
                    return False
                
                # Only check the last 50 tokens for efficiency
                last_tokens = input_ids[0, -min(50, input_ids.shape[1]):]
                
                try:
                    # Decode the sequence (the last part that might contain the stop string)
                    generated_text = self.tokenizer.decode(last_tokens)
                    
                    # Check if stop string is in the generated text
                    return self.stop_string in generated_text
                except Exception as e:
                    # If we encounter any error, log it and continue generation
                    logger.warning(f"Error in stopping criteria: {str(e)}")
                    return False
        


## Example 1

In [12]:
prob = load_math_problem(problem_idx=42)
prob

{'problem': 'The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.',
 'solution': 'Since everything in sight is even, we should begin by dividing by 2.  That gives \\[-2<x-1<4.\\] To isolate $x$, we add 1, so \\[-1<x<5.\\] Since $a=-1$ and $b=5$, we get $a+b=-1+5=\\boxed{4}$.',
 'answer': '4',
 'subject': 'Algebra',
 'level': 2,
 'unique_id': 'test/algebra/2214.json'}

In [19]:
formatted_prompt = (format_prompt(prob["problem"], [], prompt_template))
print(formatted_prompt)

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.






In [139]:
good_action_1 = '''
Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add 1 to all parts
Adding 1 to all parts, we get $1 < x < 5$ which is the form $a < x < b$.
'''

# ## Step 3: Find the sum of the endpoints
# The value of $a + b$ is $1 + 5 = 6$.


bad_action_1 = '''
Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add -1 to all parts
We get $-3 < x - 2 < 3$.


'''

In [89]:
raw_prompt = prompt_template['icl'].replace("<init_state>", "").replace("<problem_state>", "")

In [147]:
activation_list = []

In [156]:
prob = load_math_problem(problem_idx=42)

formatted_prompt = (format_prompt(prob["problem"], [], prompt_template))

formatted_prompt = good_action_1
# formatted_prompt = bad_action_1

stop_string = f"## Step 3"
temperature = 0.1

layers_to_use = [13, 14, 15]
components_to_use = ["self_attn.o_proj"]

stopping_criteria = StoppingCriteriaList([
    StopStringCriteria(policy_model.tokenizer, stop_string, formatted_prompt)
])

In [157]:
with policy_model.generate(
                formatted_prompt, 
                max_new_tokens=400,
                temperature=temperature,
                do_sample=True,
                stopping_criteria=stopping_criteria
            ) as generator:
    # Track activations
    activation_data = {}
    
    # Apply .all() to model to streamline interventions across all token generations
    policy_model.all()
    
    # Register hooks for specified layers and components
    for layer_idx in layers_to_use:
        # Get the layer
        layer = policy_model.model.layers[layer_idx]
        
        # Register hooks for each component
        for component_path in components_to_use:
            # Navigate the component path
            component = layer
            for part in component_path.split("."):
                component = getattr(component, part)
            
            # Store activations for computing steering vectors
            layer_name = f"model.layers.{layer_idx}.{component_path}"
            activation_data[layer_name] = component.output.save()
    
    # Get the generated tokens
    tokens = policy_model.generator.output.save()

activation_list.append(activation_data)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [158]:
generation = policy_model.tokenizer.decode(tokens[0], skip_special_tokens=True)            
action = generation.replace(stop_string, "").strip()
print(action)
            

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: oxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add 1 to all parts
Adding 1 to all parts, we get $1 < x < 5$ which is the form $a < x < b$.
Therefore, the value of $a + b$ is $1 + 5 = 6$.

The final answer is: $\boxed{6}$


In [159]:
test_steps = extract_steps(action, prob['problem'])

In [160]:
test_steps

['## Step 1: Divide the inequality by 2\nWe get $-2 < x - 1 < 4$.',
 '## Step 2: Add 1 to all parts\nAdding 1 to all parts, we get $1 < x < 5$ which is the form $a < x < b$.\nTherefore, the value of $a + b$ is $1 + 5 = 6$.\n\nThe final answer is: $\\boxed{6}$']

In [161]:
reward(test_steps, raw_prompt)

(0.8231921792030334, tensor([0.9025, 0.8232]))

In [162]:
bad_activation_list = []
good_activation_list = []

bad_activation_list.append(activation_list[0])
good_activation_list.append(activation_list[1])

In [165]:
steering_config = {
    "layers_to_use": [13, 14, 15],
    "components_to_use": ["self_attn.o_proj"],
    "use_layer_specific_scaling": True,
    "normalize_vectors": True,
    "steering_scale": 1.0
}

In [167]:
steering_vectors = compute_steering_vector_contrast(
    positive_activations=good_activation_list,
    negative_activations=bad_activation_list,
    layers_to_use = steering_config["layers_to_use"],
    components_to_use = steering_config["components_to_use"],
    steering_config=steering_config
)
steering_vectors

{'model.layers.13.self_attn.o_proj': tensor([[[ 0.0276, -0.0007,  0.0335,  ...,  0.0456,  0.0112,  0.0104]]],
        device='cuda:0'),
 'model.layers.14.self_attn.o_proj': tensor([[[-0.0178, -0.0199, -0.0262,  ...,  0.0092,  0.0288, -0.0107]]],
        device='cuda:0'),
 'model.layers.15.self_attn.o_proj': tensor([[[-0.0245, -0.0225,  0.0338,  ..., -0.0053,  0.0192, -0.0111]]],
        device='cuda:0')}

In [180]:
formatted_prompt = (format_prompt(prob["problem"], [], prompt_template)) + \
'''## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

'''

In [181]:
print(formatted_prompt)

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.




In [186]:
stop_string = f"## Step 3"
temperature = 0.8

stopping_criteria = StoppingCriteriaList([
    StopStringCriteria(policy_model.tokenizer, stop_string, formatted_prompt)
])

In [187]:
with policy_model.generate(
                formatted_prompt, 
                max_new_tokens=1000,
                temperature=temperature,
                do_sample=True,
                stopping_criteria=stopping_criteria
            ) as generator:
    # Track activations
    activation_data = {}
    
    # Apply .all() to model to streamline interventions across all token generations
    policy_model.all()
    
    # Register hooks for specified layers and components
    for layer_idx in layers_to_use:
        # Get the layer
        layer = policy_model.model.layers[layer_idx]
        
        # Register hooks for each component
        for component_path in components_to_use:
            # Navigate the component path
            component = layer
            for part in component_path.split("."):
                component = getattr(component, part)
            
            # Store activations for computing steering vectors
            layer_name = f"model.layers.{layer_idx}.{component_path}"
            activation_data[layer_name] = component.output.save()

            if layer_name in steering_vectors:
                component.output = component.output + steering_vectors[layer_name]
    
    # Get the generated tokens
    tokens = policy_model.generator.output.save()

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [188]:
generation = policy_model.tokenizer.decode(tokens[0], skip_special_tokens=True)            
action = generation.replace(stop_string, "").strip()
print(action)

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add 1 to all parts of the inequality
This results in $-1 < x < 5$.


In [189]:
test_steps = extract_steps(action, prob['problem'])
reward(test_steps, raw_prompt)

(0.9291694760322571, tensor([0.9025, 0.9292]))

## Example 2

In [1]:
prob = load_math_problem(problem_idx=87)
prob

NameError: name 'load_math_problem' is not defined

In [ ]:
formatted_prompt = (format_prompt(prob["problem"], [], prompt_template))
print(formatted_prompt)

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.






In [ ]:
good_action_1 = '''
Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add 1 to all parts
Adding 1 to all parts, we get $1 < x < 5$ which is the form $a < x < b$.
'''

# ## Step 3: Find the sum of the endpoints
# The value of $a + b$ is $1 + 5 = 6$.


bad_action_1 = '''
Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add -1 to all parts
We get $-3 < x - 2 < 3$.


'''

In [ ]:
raw_prompt = prompt_template['icl'].replace("<init_state>", "").replace("<problem_state>", "")

In [ ]:
activation_list = []

In [ ]:
prob = load_math_problem(problem_idx=42)

formatted_prompt = (format_prompt(prob["problem"], [], prompt_template))

formatted_prompt = good_action_1
# formatted_prompt = bad_action_1

stop_string = f"## Step 3"
temperature = 0.1

layers_to_use = [13, 14, 15]
components_to_use = ["self_attn.o_proj"]

stopping_criteria = StoppingCriteriaList([
    StopStringCriteria(policy_model.tokenizer, stop_string, formatted_prompt)
])

In [ ]:
with policy_model.generate(
                formatted_prompt, 
                max_new_tokens=400,
                temperature=temperature,
                do_sample=True,
                stopping_criteria=stopping_criteria
            ) as generator:
    # Track activations
    activation_data = {}
    
    # Apply .all() to model to streamline interventions across all token generations
    policy_model.all()
    
    # Register hooks for specified layers and components
    for layer_idx in layers_to_use:
        # Get the layer
        layer = policy_model.model.layers[layer_idx]
        
        # Register hooks for each component
        for component_path in components_to_use:
            # Navigate the component path
            component = layer
            for part in component_path.split("."):
                component = getattr(component, part)
            
            # Store activations for computing steering vectors
            layer_name = f"model.layers.{layer_idx}.{component_path}"
            activation_data[layer_name] = component.output.save()
    
    # Get the generated tokens
    tokens = policy_model.generator.output.save()

activation_list.append(activation_data)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
generation = policy_model.tokenizer.decode(tokens[0], skip_special_tokens=True)            
action = generation.replace(stop_string, "").strip()
print(action)
            

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: oxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add 1 to all parts
Adding 1 to all parts, we get $1 < x < 5$ which is the form $a < x < b$.
Therefore, the value of $a + b$ is $1 + 5 = 6$.

The final answer is: $\boxed{6}$


In [ ]:
test_steps = extract_steps(action, prob['problem'])

In [ ]:
test_steps

['## Step 1: Divide the inequality by 2\nWe get $-2 < x - 1 < 4$.',
 '## Step 2: Add 1 to all parts\nAdding 1 to all parts, we get $1 < x < 5$ which is the form $a < x < b$.\nTherefore, the value of $a + b$ is $1 + 5 = 6$.\n\nThe final answer is: $\\boxed{6}$']

In [ ]:
reward(test_steps, raw_prompt)

(0.8231921792030334, tensor([0.9025, 0.8232]))

In [ ]:
bad_activation_list = []
good_activation_list = []

bad_activation_list.append(activation_list[0])
good_activation_list.append(activation_list[1])

In [ ]:
steering_config = {
    "layers_to_use": [13, 14, 15],
    "components_to_use": ["self_attn.o_proj"],
    "use_layer_specific_scaling": True,
    "normalize_vectors": True,
    "steering_scale": 1.0
}

In [ ]:
steering_vectors = compute_steering_vector_contrast(
    positive_activations=good_activation_list,
    negative_activations=bad_activation_list,
    layers_to_use = steering_config["layers_to_use"],
    components_to_use = steering_config["components_to_use"],
    steering_config=steering_config
)
steering_vectors

{'model.layers.13.self_attn.o_proj': tensor([[[ 0.0276, -0.0007,  0.0335,  ...,  0.0456,  0.0112,  0.0104]]],
        device='cuda:0'),
 'model.layers.14.self_attn.o_proj': tensor([[[-0.0178, -0.0199, -0.0262,  ...,  0.0092,  0.0288, -0.0107]]],
        device='cuda:0'),
 'model.layers.15.self_attn.o_proj': tensor([[[-0.0245, -0.0225,  0.0338,  ..., -0.0053,  0.0192, -0.0111]]],
        device='cuda:0')}

In [ ]:
formatted_prompt = (format_prompt(prob["problem"], [], prompt_template)) + \
'''## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

'''

In [ ]:
print(formatted_prompt)

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.




In [ ]:
stop_string = f"## Step 3"
temperature = 0.8

stopping_criteria = StoppingCriteriaList([
    StopStringCriteria(policy_model.tokenizer, stop_string, formatted_prompt)
])

In [ ]:
with policy_model.generate(
                formatted_prompt, 
                max_new_tokens=1000,
                temperature=temperature,
                do_sample=True,
                stopping_criteria=stopping_criteria
            ) as generator:
    # Track activations
    activation_data = {}
    
    # Apply .all() to model to streamline interventions across all token generations
    policy_model.all()
    
    # Register hooks for specified layers and components
    for layer_idx in layers_to_use:
        # Get the layer
        layer = policy_model.model.layers[layer_idx]
        
        # Register hooks for each component
        for component_path in components_to_use:
            # Navigate the component path
            component = layer
            for part in component_path.split("."):
                component = getattr(component, part)
            
            # Store activations for computing steering vectors
            layer_name = f"model.layers.{layer_idx}.{component_path}"
            activation_data[layer_name] = component.output.save()

            if layer_name in steering_vectors:
                component.output = component.output + steering_vectors[layer_name]
    
    # Get the generated tokens
    tokens = policy_model.generator.output.save()

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
generation = policy_model.tokenizer.decode(tokens[0], skip_special_tokens=True)            
action = generation.replace(stop_string, "").strip()
print(action)

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

The solution to $-4 < 2(x - 1) < 8$ is expressed in the form $a < x < b$. Find the value of $a + b$.



## Step 1: Divide the inequality by 2
We get $-2 < x - 1 < 4$.

## Step 2: Add 1 to all parts of the inequality
This results in $-1 < x < 5$.


In [ ]:
test_steps = extract_steps(action, prob['problem'])
reward(test_steps, raw_prompt)

(0.9291694760322571, tensor([0.9025, 0.9292]))